In [4]:
import numpy as np
from dataclasses import dataclass
from math import exp, log, sqrt, erf

In [5]:
# -----------------------------
# 1) Model + simulation (from before, kept intact)
# -----------------------------
@dataclass(frozen=True)
class RegimeParams:
    # States: 0 = low vol, 1 = high vol
    sigma: np.ndarray  # shape (2,)
    P: np.ndarray      # shape (2,2)

@dataclass(frozen=True)
class MarketParams:
    r: float   # cont. comp
    q: float   # cont. dividend yield

def validate_params(regime: RegimeParams, market: MarketParams):
    sigma = np.asarray(regime.sigma, dtype=float)
    P = np.asarray(regime.P, dtype=float)

    if sigma.shape != (2,):
        raise ValueError("regime.sigma must have shape (2,)")
    if np.any(sigma <= 0):
        raise ValueError("All sigma values must be > 0")

    if P.shape != (2, 2):
        raise ValueError("regime.P must have shape (2,2)")
    if np.any(P < 0):
        raise ValueError("Transition probabilities must be >= 0")
    if not np.allclose(P.sum(axis=1), 1.0, atol=1e-12):
        raise ValueError("Each row of transition matrix P must sum to 1")

    if not (-0.05 <= market.r <= 0.20):
        raise ValueError("market.r seems out of a reasonable range")
    if not (0.0 <= market.q <= 0.20):
        raise ValueError("market.q must be in [0, 0.20] for this setup")

def simulate_markov_regimes(P: np.ndarray, n_paths: int, n_steps: int, x0: int = 0, seed: int = 42):
    rng = np.random.default_rng(seed)
    P = np.asarray(P, dtype=float)

    X = np.empty((n_paths, n_steps + 1), dtype=np.int8)
    X[:, 0] = x0

    U = rng.random((n_paths, n_steps))
    p_to_0 = P[:, 0]
    for t in range(n_steps):
        prev = X[:, t]
        thr = p_to_0[prev]
        X[:, t + 1] = (U[:, t] >= thr).astype(np.int8)

    return X

def simulate_asset_paths_regime_switching(
    S0: float,
    T: float,
    n_steps: int,
    n_paths: int,
    market: MarketParams,
    regime: RegimeParams,
    x0: int = 0,
    seed: int = 42
):
    validate_params(regime, market)
    if S0 <= 0:
        raise ValueError("S0 must be > 0")
    if T <= 0:
        raise ValueError("T must be > 0")
    if n_steps < 1:
        raise ValueError("n_steps must be >= 1")
    if n_paths < 1:
        raise ValueError("n_paths must be >= 1")

    dt = T / n_steps
    t_grid = np.linspace(0.0, T, n_steps + 1)

    X = simulate_markov_regimes(regime.P, n_paths, n_steps, x0=x0, seed=seed)

    rng = np.random.default_rng(seed + 1)
    Z = rng.standard_normal((n_paths, n_steps))

    S = np.empty((n_paths, n_steps + 1), dtype=float)
    S[:, 0] = S0

    r, q = market.r, market.q
    sigma_states = np.asarray(regime.sigma, dtype=float)

    for t in range(n_steps):
        sig = sigma_states[X[:, t]]
        drift = (r - q - 0.5 * sig**2) * dt
        diff = sig * np.sqrt(dt) * Z[:, t]
        S[:, t + 1] = S[:, t] * np.exp(drift + diff)

    return t_grid, S, X


In [6]:
# -----------------------------
# 2) European Call pricing by Monte Carlo (+SE +CI)
# -----------------------------
def price_european_call_mc(S_T: np.ndarray, K: float, r: float, T: float):
    """
    Price: e^{-rT} E[(S_T - K)+]
    Returns: price, std_error, CI95_low, CI95_high
    """
    payoff = np.maximum(S_T - K, 0.0)
    disc_payoff = np.exp(-r * T) * payoff

    price = disc_payoff.mean()
    # sample std (ddof=1) for MC error
    s = disc_payoff.std(ddof=1)
    se = s / np.sqrt(disc_payoff.size)
    ci_low = price - 1.96 * se
    ci_high = price + 1.96 * se
    return price, se, ci_low, ci_high

In [7]:
# -----------------------------
# 3) Black–Scholes (continuous dividend yield q) for sanity check
# -----------------------------
def _norm_cdf(x: float) -> float:
    # CDF of standard normal via erf
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

def black_scholes_call(S0: float, K: float, r: float, q: float, sigma: float, T: float) -> float:
    if T <= 0:
        return max(S0 - K, 0.0)
    if sigma <= 0:
        # deterministic forward under risk-neutral: S_T = S0*exp((r-q)T)
        ST = S0 * exp((r - q) * T)
        return exp(-r * T) * max(ST - K, 0.0)

    d1 = (log(S0 / K) + (r - q + 0.5 * sigma * sigma) * T) / (sigma * sqrt(T))
    d2 = d1 - sigma * sqrt(T)
    return S0 * exp(-q * T) * _norm_cdf(d1) - K * exp(-r * T) * _norm_cdf(d2)

In [8]:
# -----------------------------
# 4) End-to-end experiment (ITM/ATM/OTM) + BS check with constant vol
# -----------------------------
def run_european_mc_experiment(
    S0=100.0,
    T=1.0,
    n_steps=252,
    n_paths=20000,
    r=0.03,
    q=0.02,
    sigma_L=0.18,
    sigma_H=0.40,
    P=np.array([[0.95, 0.05],
                [0.10, 0.90]]),
    strikes=(90.0, 100.0, 110.0),
    seed=123
):
    market = MarketParams(r=r, q=q)
    regime = RegimeParams(sigma=np.array([sigma_L, sigma_H], dtype=float), P=np.array(P, dtype=float))

    t, S, X = simulate_asset_paths_regime_switching(
        S0=S0, T=T, n_steps=n_steps, n_paths=n_paths,
        market=market, regime=regime, x0=0, seed=seed
    )

    S_T = S[:, -1]

    # --- Core MC results under regime switching ---
    results = []
    for K in strikes:
        price, se, lo, hi = price_european_call_mc(S_T, K, r=r, T=T)
        results.append({
            "K": K,
            "MC_price": price,
            "MC_SE": se,
            "MC_CI95_low": lo,
            "MC_CI95_high": hi
        })

    # --- Sanity check: constant vol -> compare MC vs Black–Scholes
    # We set sigma_L = sigma_H = sigma_const and P irrelevant (still works)
    sigma_const = sigma_L  # choose low vol as constant reference
    regime_const = RegimeParams(sigma=np.array([sigma_const, sigma_const], dtype=float), P=np.array(P, dtype=float))

    t2, S2, X2 = simulate_asset_paths_regime_switching(
        S0=S0, T=T, n_steps=n_steps, n_paths=n_paths,
        market=market, regime=regime_const, x0=0, seed=seed + 999
    )
    S_T_const = S2[:, -1]

    sanity = []
    for K in strikes:
        mc_p, mc_se, mc_lo, mc_hi = price_european_call_mc(S_T_const, K, r=r, T=T)
        bs_p = black_scholes_call(S0, K, r, q, sigma_const, T)
        sanity.append({
            "K": K,
            "MC_constvol_price": mc_p,
            "MC_constvol_SE": mc_se,
            "MC_constvol_CI95_low": mc_lo,
            "MC_constvol_CI95_high": mc_hi,
            "BS_price": bs_p,
            "MC_minus_BS": mc_p - bs_p
        })

    # Quick extra checks (expectations)
    expected_ST = S0 * np.exp((r - q) * T)
    st_mean = S_T.mean()
    st_std = S_T.std(ddof=1)

    regime_props_T = np.bincount(X[:, -1], minlength=2) / X.shape[0]

    return {
        "paths_info": {
            "S_shape": S.shape,
            "X_shape": X.shape,
            "E_ST_theoretical": expected_ST,
            "E_ST_MC": st_mean,
            "ST_std_MC": st_std,
            "regime_proportions_T": regime_props_T
        },
        "regime_switching_results": results,
        "constant_vol_sanity_check": sanity
    }

In [9]:
# -----------------------------
# 5) Run
# -----------------------------
if __name__ == "__main__":
    out = run_european_mc_experiment(
        S0=100.0, T=1.0, n_steps=252, n_paths=20000,
        r=0.03, q=0.02,
        sigma_L=0.18, sigma_H=0.40,
        P=np.array([[0.95, 0.05],
                    [0.10, 0.90]]),
        strikes=(90.0, 100.0, 110.0),
        seed=123
    )

    print("=== CORE SIMULATION CHECKS ===")
    pi = out["paths_info"]
    print("S shape:", pi["S_shape"], "X shape:", pi["X_shape"])
    print(f"E[S_T] theoretical: {pi['E_ST_theoretical']:.6f}")
    print(f"E[S_T] MC        : {pi['E_ST_MC']:.6f}")
    print(f"Std[S_T] MC      : {pi['ST_std_MC']:.6f}")
    print("Regime proportions at T (state0,state1):", pi["regime_proportions_T"])

    print("\n=== EUROPEAN CALL PRICES (REGIME SWITCHING) ===")
    for row in out["regime_switching_results"]:
        print(f"K={row['K']:.0f}  MC={row['MC_price']:.6f}  SE={row['MC_SE']:.6f}  "
              f"CI95=[{row['MC_CI95_low']:.6f},{row['MC_CI95_high']:.6f}]")

    print("\n=== SANITY CHECK (CONSTANT VOL): MC vs BLACK–SCHOLES ===")
    for row in out["constant_vol_sanity_check"]:
        print(f"K={row['K']:.0f}  MC={row['MC_constvol_price']:.6f} (SE={row['MC_constvol_SE']:.6f})  "
              f"BS={row['BS_price']:.6f}  MC-BS={row['MC_minus_BS']:.6f}")

=== CORE SIMULATION CHECKS ===
S shape: (20000, 253) X shape: (20000, 253)
E[S_T] theoretical: 101.005017
E[S_T] MC        : 100.876307
Std[S_T] MC      : 28.075037
Regime proportions at T (state0,state1): [0.6654 0.3346]

=== EUROPEAN CALL PRICES (REGIME SWITCHING) ===
K=90  MC=16.163324  SE=0.153404  CI95=[15.862651,16.463996]
K=100  MC=10.983784  SE=0.132328  CI95=[10.724421,11.243147]
K=110  MC=7.200905  SE=0.110586  CI95=[6.984156,7.417653]

=== SANITY CHECK (CONSTANT VOL): MC vs BLACK–SCHOLES ===
K=90  MC=13.178149 (SE=0.102931)  BS=13.300945  MC-BS=-0.122796
K=100  MC=7.386450 (SE=0.082135)  BS=7.492725  MC-BS=-0.106275
K=110  MC=3.694991 (SE=0.059749)  BS=3.784439  MC-BS=-0.089448


# Código LSM (americana) con regresión separada por régimen

In [21]:
import numpy as np
import time
from dataclasses import dataclass

@dataclass(frozen=True)
class LSMResult:
    price: float
    se: float
    ci95_low: float
    ci95_high: float
    elapsed_sec: float
    per_path_pv: np.ndarray
    exercise_boundary: dict  # {state: np.ndarray}

def _make_basis(x: np.ndarray, degree: int = 2) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    cols = [np.ones_like(x)]
    for d in range(1, degree + 1):
        cols.append(x ** d)
    return np.column_stack(cols)

def price_american_call_lsm_regime_cf(
    S: np.ndarray,
    X: np.ndarray,
    K: float,
    r: float,
    T: float,
    basis_degree: int = 2,
    by_regime: bool = True,
    scale_by_strike: bool = True,
    min_reg_points: int = 200
) -> LSMResult:
    import numpy as np
    import time

    t0 = time.perf_counter()

    S = np.asarray(S, dtype=float)
    X = np.asarray(X)
    if S.shape != X.shape:
        raise ValueError("S and X must have the same shape (M, N+1).")
    M, Np1 = S.shape
    N = Np1 - 1
    dt = T / N

    payoff = np.maximum(S - K, 0.0)

    CF = payoff[:, -1].copy()
    tau = np.full(M, N, dtype=int)  # exercise time index (default maturity)

    boundary = {0: np.full(Np1, np.nan), 1: np.full(Np1, np.nan)}

    for t in range(N - 1, 0, -1):
        # ✅ FIX: a path is "alive" at time t if its current exercise time is after t
        alive = (tau > t)
        if not np.any(alive):
            break

        S_t = S[alive, t]
        X_t = X[alive, t]
        payoff_t = payoff[alive, t]

        itm = payoff_t > 0
        if not np.any(itm):
            continue

        tau_alive = tau[alive][itm]
        CF_alive = CF[alive][itm]
        Y = CF_alive * np.exp(-r * (tau_alive - t) * dt)

        S_itm = S_t[itm]
        X_itm = X_t[itm]
        payoff_itm = payoff_t[itm]

        x_feat = (S_itm / K) if scale_by_strike else S_itm
        C_hat = np.zeros_like(Y)

        if by_regime:
            for state in (0, 1):
                mask = (X_itm == state)
                if mask.sum() < min_reg_points:
                    if mask.sum() > 0:
                        C_hat[mask] = Y[mask].mean()
                    continue
                A = _make_basis(x_feat[mask], degree=basis_degree)
                beta, *_ = np.linalg.lstsq(A, Y[mask], rcond=None)
                C_hat[mask] = A @ beta
        else:
            if x_feat.size >= min_reg_points:
                A = _make_basis(x_feat, degree=basis_degree)
                beta, *_ = np.linalg.lstsq(A, Y, rcond=None)
                C_hat = A @ beta
            else:
                C_hat[:] = Y.mean()

        exercise = payoff_itm >= C_hat

        alive_idx = np.flatnonzero(alive)
        itm_idx_global = alive_idx[np.flatnonzero(itm)]
        ex_global = itm_idx_global[exercise]

        CF[ex_global] = payoff[ex_global, t]
        tau[ex_global] = t

        for state in (0, 1):
            mask_state = (X_itm == state)
            if np.any(mask_state):
                S_state = S_itm[mask_state]
                ex_state = exercise[mask_state]
                if np.any(ex_state):
                    boundary[state][t] = np.nanmin(S_state[ex_state])

    PV0 = CF * np.exp(-r * tau * dt)
    payoff0 = max(S[0, 0] - K, 0.0)
    PV0 = np.maximum(PV0, payoff0)

    price = PV0.mean()
    s = PV0.std(ddof=1)
    se = s / np.sqrt(M)
    ci_low = price - 1.96 * se
    ci_high = price + 1.96 * se
    elapsed = time.perf_counter() - t0

    return LSMResult(
        price=price,
        se=se,
        ci95_low=ci_low,
        ci95_high=ci_high,
        elapsed_sec=elapsed,
        per_path_pv=PV0,
        exercise_boundary=boundary
    )

# -----------------------------
# Example usage (plug into your existing simulation output)
# -----------------------------
if __name__ == "__main__":
    # Assume you already have t, S, X from your simulator
    # Here I just show how you'd call it:
    # result = price_american_call_lsm_regime(S, X, K=100.0, r=0.03, T=1.0, basis_degree=2, by_regime=True)
    # print(result)
    pass


In [11]:
K = 100.0
res = price_american_call_lsm_regime(
    S=S, X=X, K=K, r=0.03, T=1.0,
    basis_degree=2,
    by_regime=True,
    scale_by_strike=True,
    min_reg_points=200
)

print(f"LSM American Call (K={K:.0f}): {res.price:.6f}")
print(f"SE: {res.se:.6f}  CI95=[{res.ci95_low:.6f},{res.ci95_high:.6f}]")
print(f"Runtime: {res.elapsed_sec:.3f} sec")


LSM American Call (K=100): 10.933603
SE: 0.107119  CI95=[10.723649,11.143556]
Runtime: 0.488 sec


In [12]:
b0 = res.exercise_boundary[0]  # régimen bajo
b1 = res.exercise_boundary[1]  # régimen alto
print("Boundary (low vol) first non-nan:", np.nanmin(b0))
print("Boundary (high vol) first non-nan:", np.nanmin(b1))


Boundary (low vol) first non-nan: 123.61837302547639
Boundary (high vol) first non-nan: 118.93020618227209


In [13]:
Ks = [90.0, 100.0, 110.0]

for K in Ks:
    res = price_american_call_lsm_regime(
        S=S, X=X, K=K, r=0.03, T=1.0,
        basis_degree=2,
        by_regime=True,
        scale_by_strike=True,
        min_reg_points=200
    )
    print(f"\n=== LSM American Call (K={K:.0f}) ===")
    print(f"Price: {res.price:.6f}")
    print(f"SE: {res.se:.6f}  CI95=[{res.ci95_low:.6f},{res.ci95_high:.6f}]")
    print(f"Runtime: {res.elapsed_sec:.3f} sec")
    print("Boundary (low vol) first non-nan:", np.nanmin(res.exercise_boundary[0]))
    print("Boundary (high vol) first non-nan:", np.nanmin(res.exercise_boundary[1]))



=== LSM American Call (K=90) ===
Price: 20.282137
SE: 0.083832  CI95=[20.117827,20.446447]
Runtime: 0.430 sec
Boundary (low vol) first non-nan: 113.27682576409977
Boundary (high vol) first non-nan: 110.86876521461787

=== LSM American Call (K=100) ===
Price: 10.933603
SE: 0.107119  CI95=[10.723649,11.143556]
Runtime: 0.374 sec
Boundary (low vol) first non-nan: 123.61837302547639
Boundary (high vol) first non-nan: 118.93020618227209

=== LSM American Call (K=110) ===
Price: 7.228826
SE: 0.094213  CI95=[7.044169,7.413484]
Runtime: 0.277 sec
Boundary (low vol) first non-nan: 124.28885819779318
Boundary (high vol) first non-nan: 122.76416743942978


In [14]:
for K in Ks:
    eu_price, eu_se, eu_lo, eu_hi = price_european_call_mc(S[:, -1], K, r=0.03, T=1.0)
    print(f"\n--- European MC (K={K:.0f}) ---")
    print(f"Price: {eu_price:.6f}")
    print(f"SE: {eu_se:.6f}  CI95=[{eu_lo:.6f},{eu_hi:.6f}]")



--- European MC (K=90) ---
Price: 16.163324
SE: 0.153404  CI95=[15.862651,16.463996]

--- European MC (K=100) ---
Price: 10.983784
SE: 0.132328  CI95=[10.724421,11.243147]

--- European MC (K=110) ---
Price: 7.200905
SE: 0.110586  CI95=[6.984156,7.417653]


In [15]:
for deg in [2, 3]:
    res = price_american_call_lsm_regime(
        S=S, X=X, K=100.0, r=0.03, T=1.0,
        basis_degree=deg,
        by_regime=True,
        scale_by_strike=True,
        min_reg_points=200
    )
    print(f"\n*** Stability check (K=100) degree={deg} ***")
    print(f"Price: {res.price:.6f}  SE: {res.se:.6f}  Runtime: {res.elapsed_sec:.3f}s")



*** Stability check (K=100) degree=2 ***
Price: 10.933603  SE: 0.107119  Runtime: 0.371s

*** Stability check (K=100) degree=3 ***
Price: 10.946520  SE: 0.108235  Runtime: 0.425s


Miramos el problema de que la Americana da mucho más alto que la Europea


In [16]:
# Re-simulate with larger M for european only
t_big, S_big, X_big = simulate_asset_paths_regime_switching(
    S0=100.0, T=1.0, n_steps=252, n_paths=200000,
    market=MarketParams(r=0.03, q=0.02),
    regime=RegimeParams(sigma=np.array([0.18, 0.40]), P=np.array([[0.95,0.05],[0.10,0.90]])),
    x0=0, seed=777
)

for K in [90.0, 100.0, 110.0]:
    eu_p, eu_se, eu_lo, eu_hi = price_european_call_mc(S_big[:, -1], K, r=0.03, T=1.0)
    print(f"EUROPEAN (M=200k) K={K:.0f}: {eu_p:.6f}  SE={eu_se:.6f}  CI=[{eu_lo:.6f},{eu_hi:.6f}]")


EUROPEAN (M=200k) K=90: 16.248970  SE=0.048509  CI=[16.153893,16.344047]
EUROPEAN (M=200k) K=100: 11.024042  SE=0.041886  CI=[10.941945,11.106139]
EUROPEAN (M=200k) K=110: 7.225688  SE=0.035018  CI=[7.157053,7.294323]


In [22]:
# Re-simulate with q=0
t0, S0q, X0q = simulate_asset_paths_regime_switching(
    S0=100.0, T=1.0, n_steps=252, n_paths=50000,
    market=MarketParams(r=0.03, q=0.0),
    regime=RegimeParams(sigma=np.array([0.18, 0.40]), P=np.array([[0.95,0.05],[0.10,0.90]])),
    x0=0, seed=888
)

for K in [90.0, 100.0, 110.0]:
    eu_p, eu_se, *_ = price_european_call_mc(S0q[:, -1], K, r=0.03, T=1.0)
    am = price_american_call_lsm_regime_cf(S0q, X0q, K=K, r=0.03, T=1.0, basis_degree=2, by_regime=True)
    print(f"K={K:.0f}  EURO={eu_p:.6f} (SE={eu_se:.6f})  AM={am.price:.6f} (SE={am.se:.6f})")



K=90  EURO=17.633350 (SE=0.101259)  AM=21.593836 (SE=0.065967)
K=100  EURO=12.150401 (SE=0.088244)  AM=11.950146 (SE=0.073994)
K=110  EURO=8.069116 (SE=0.074530)  AM=7.960980 (SE=0.063919)


In [20]:
for K in [90.0, 100.0, 110.0]:
    eu_p, eu_se, eu_lo, eu_hi = price_european_call_mc(S[:, -1], K, r=0.03, T=1.0)
    am = price_american_call_lsm_regime_cf(S, X, K=K, r=0.03, T=1.0, basis_degree=2, by_regime=True)
    print(f"\nK={K:.0f}")
    print(f"EU  {eu_p:.6f}  SE={eu_se:.6f}  CI=[{eu_lo:.6f},{eu_hi:.6f}]")
    print(f"AM  {am.price:.6f}  SE={am.se:.6f}  CI=[{am.ci95_low:.6f},{am.ci95_high:.6f}]")



K=90
EU  16.163324  SE=0.153404  CI=[15.862651,16.463996]
AM  20.855098  SE=0.131191  CI=[20.597962,21.112233]

K=100
EU  10.983784  SE=0.132328  CI=[10.724421,11.243147]
AM  11.709313  SE=0.129221  CI=[11.456040,11.962586]

K=110
EU  7.200905  SE=0.110586  CI=[6.984156,7.417653]
AM  7.745882  SE=0.108857  CI=[7.532522,7.959242]
